# Precios de coches de segunda mano en España

Cinco gráficos con datos de Kaggle (`gandpablo/cars-prices-for-machine-learning`). Primero Matplotlib, luego los mismos con Seaborn, y al final la página con Streamlit.

El dataset incluye marca, modelo, precio, etiqueta ambiental, kilómetros, dueño único, CV, combustible, transmisión y más.


## 1. Descarga del conjunto de datos desde Kaggle

Para iniciar el análisis, utilizaremos la API oficial de Kaggle para descargar el archivo de precios de coches usados en España directamente en el entorno.


En [kaggle.com/settings/api](https://www.kaggle.com/settings/api), pulsa **Create Legacy API Key** y sube aquí el `kaggle.json`.


In [ ]:
!pip install -q kaggle seaborn streamlit


In [ ]:
from google.colab import files
from pathlib import Path
import os

uploaded = files.upload()

destino = Path.home() / ".kaggle"
destino.mkdir(parents=True, exist_ok=True)
nombre = next(iter(uploaded))
texto = uploaded[nombre].decode("utf-8").strip()

if texto.startswith("{") and "key" in texto:
    ruta = destino / "kaggle.json"
    ruta.write_bytes(uploaded[nombre])
elif texto.startswith("KGAT_"):
    ruta = destino / "access_token"
    ruta.write_text(texto + "\n", encoding="utf-8")
else:
    raise SystemExit("Sube el kaggle.json de Kaggle.")

os.chmod(ruta, 0o600)


In [ ]:
!mkdir -p data
!kaggle datasets download -d gandpablo/cars-prices-for-machine-learning -p data --unzip


## 2. Carga y preprocesamiento de los datos

Importamos las librerías (`pandas`, `matplotlib` y `seaborn`) y limpiamos el conjunto. Convertimos a numérico las columnas de precio, kilómetros y potencia, filtramos valores absurdos y traducimos combustible, transmisión y etiqueta ambiental para facilitar la lectura.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

SURFACE = "#fcfcfb"
TEXT, TEXT2, GRID = "#0b0b0b", "#52514e", "#e3e2de"
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT2, "xtick.color": TEXT2, "ytick.color": TEXT2,
    "text.color": TEXT, "axes.titlelocation": "left", "axes.titleweight": "semibold",
    "axes.titlesize": 12, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 1,
    "lines.linewidth": 2, "lines.solid_capstyle": "round", "legend.frameon": False,
    "font.size": 10, "figure.dpi": 110,
})


def xgrid(ax):
    ax.grid(False)
    ax.grid(True, axis="x")


def ygrid(ax):
    ax.grid(False)
    ax.grid(True, axis="y")


ruta_csv = Path("data/Cars_Data_6k.csv")
if not ruta_csv.exists():
    candidatos = list(Path("data").glob("*.csv"))
    if not candidatos:
        raise FileNotFoundError("No hay CSV en data/. Ejecuta antes la descarga de Kaggle.")
    ruta_csv = candidatos[0]

df = pd.read_csv(ruta_csv)
df.columns = [c.strip() for c in df.columns]

for col in ["Price", "KM", "CV", "Year"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Brand", "Price", "KM", "Year", "Fuel", "Transmission", "Sticker"]).copy()
df = df[(df["Price"] > 500) & (df["Price"] < 150_000)]
df = df[(df["KM"] >= 0) & (df["KM"] < 400_000)]
df = df[(df["Year"] >= 2000) & (df["Year"] <= 2024)]

COMBUSTIBLE = {
    "Gasolina": "Gasolina",
    "Diésel": "Diésel",
    "Diesel": "Diésel",
    "Híbrido": "Híbrido",
    "Hibrido": "Híbrido",
    "Híbrido Enchufable": "Híbrido enchufable",
    "Eléctrico": "Eléctrico",
    "Electrico": "Eléctrico",
    "GLP": "GLP",
    "GNC": "GNC",
}
TRANSMISION = {"MANUAL": "Manual", "AUTO": "Automático", "AUTOMATICO": "Automático"}
ETIQUETA = {
    "0_EMISIONES": "0 emisiones",
    "ECO": "ECO",
    "C": "C",
    "B": "B",
    "SIN DISTINTIVO": "Sin distintivo",
}

df["Combustible"] = df["Fuel"].map(COMBUSTIBLE).fillna(df["Fuel"])
df["Transmision"] = df["Transmission"].astype(str).str.upper().map(TRANSMISION).fillna(df["Transmission"])
df["Etiqueta"] = df["Sticker"].astype(str).str.upper().map(
    {k.upper(): v for k, v in ETIQUETA.items()}
).fillna(df["Sticker"].astype(str))
df["Un_dueno"] = df["One_owner"].astype(str).str.lower().isin(["true", "1", "yes"])

top_marcas = df["Brand"].value_counts().head(10).index
por_marca = (
    df[df["Brand"].isin(top_marcas)]
    .groupby("Brand", as_index=False)["Price"]
    .mean()
    .sort_values("Price", ascending=False)
)
por_anio = (
    df.groupby("Year", as_index=False)["Price"]
    .mean()
    .sort_values("Year")
)
muestra = df.sample(n=min(800, len(df)), random_state=42)
por_combustible = df["Combustible"].value_counts()
combustibles = por_combustible.index.tolist()

print(f"Coches tras limpieza: {len(df):,}".replace(",", "."))
df.head()


## Matplotlib


### Barras


Precio medio de las 10 marcas con más anuncios, para comparar categorías de forma directa.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
barras = ax.barh(por_marca["Brand"], por_marca["Price"], color=PAL[0], height=0.62)
ax.invert_yaxis()
ax.bar_label(barras, fmt="%.0f €", padding=3, color=TEXT, fontsize=8)
ax.set_xlim(0, por_marca["Price"].max() * 1.18)
ax.set_xlabel("Precio medio (€)")
xgrid(ax)
ax.set_title("Precio medio por marca (top 10)")
fig.tight_layout()
plt.show()


### Línea


Evolución del precio medio según el año de matriculación: los coches más nuevos tienden a costar más.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(por_anio["Year"], por_anio["Price"], color=PAL[0])
ultimo_x = por_anio["Year"].iloc[-1]
ultimo_y = por_anio["Price"].iloc[-1]
ax.plot(ultimo_x, ultimo_y, "o", color=PAL[0], markersize=8, markeredgecolor=SURFACE, markeredgewidth=2)
ax.annotate(f"{ultimo_y:,.0f} €".replace(",", "."), (ultimo_x, ultimo_y), xytext=(8, 0), textcoords="offset points", va="center", color=TEXT)
ax.set_xticks(por_anio["Year"][::3])
ax.set_xlabel("Año de matriculación")
ax.set_ylabel("Precio medio (€)")
ygrid(ax)
ax.set_title("Precio medio por año")
fig.tight_layout()
plt.show()


### Dispersión


Relación entre kilómetros y precio: a más kilometraje, el precio suele bajar. El color distingue el combustible.


In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.6))
colores = {nombre: PAL[i % len(PAL)] for i, nombre in enumerate(combustibles)}
for nombre in combustibles:
    grupo = muestra[muestra["Combustible"] == nombre]
    ax.scatter(
        grupo["KM"], grupo["Price"],
        s=28, color=colores[nombre], edgecolor=SURFACE, linewidth=0.8, label=nombre, alpha=0.75,
    )
ax.set_xlabel("Kilómetros")
ax.set_ylabel("Precio (€)")
ax.set_title("Kilómetros frente a precio")
ax.legend(title="Combustible", loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()


### Histograma


Distribución del precio de los anuncios: la mayoría se concentra en tramos medios-bajos.


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.hist(df["Price"], bins=30, color=PAL[0], edgecolor=SURFACE, linewidth=1.2)
ax.set_xlabel("Precio (€)")
ax.set_ylabel("Coches")
ygrid(ax)
ax.set_title("Distribución de precios")
fig.tight_layout()
plt.show()


### Tarta


Reparto de anuncios según el tipo de combustible (gasolina, diésel, híbrido, etc.).


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 5.4))
ax.pie(
    por_combustible.values,
    labels=[f"{nombre}\n{valor}" for nombre, valor in por_combustible.items()],
    colors=PAL[: len(por_combustible)],
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": SURFACE, "linewidth": 2},
    textprops={"color": TEXT, "fontsize": 8},
)
ax.axis("equal")
ax.grid(False)
ax.set_title("Anuncios por combustible")
fig.tight_layout()
plt.show()


## Seaborn


## 3. Réplica de los gráficos con Seaborn

A continuación, replicamos la misma estructura de análisis utilizando Seaborn para aprovechar su estilo visual y sus funciones estadísticas.


In [ ]:
sns.set_theme(style="whitegrid", palette=PAL)
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT2, "xtick.color": TEXT2, "ytick.color": TEXT2,
    "text.color": TEXT, "axes.titlelocation": "left", "axes.titleweight": "semibold",
    "axes.titlesize": 12, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "font.size": 10, "figure.dpi": 110,
})


### Barras


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
sns.barplot(data=por_marca, y="Brand", x="Price", color=PAL[0], order=por_marca["Brand"], ax=ax)
ax.set_xlim(0, por_marca["Price"].max() * 1.12)
ax.set_xlabel("Precio medio (€)")
ax.set_ylabel("")
xgrid(ax)
ax.set_title("Precio medio por marca (top 10)")
fig.tight_layout()
plt.show()


### Línea


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
sns.lineplot(data=por_anio, x="Year", y="Price", color=PAL[0], ax=ax)
ultimo_x = por_anio["Year"].iloc[-1]
ultimo_y = por_anio["Price"].iloc[-1]
ax.plot(ultimo_x, ultimo_y, "o", color=PAL[0], markersize=8, markeredgecolor=SURFACE, markeredgewidth=2)
ax.annotate(f"{ultimo_y:,.0f} €".replace(",", "."), (ultimo_x, ultimo_y), xytext=(8, 0), textcoords="offset points", va="center", color=TEXT)
ax.set_xticks(por_anio["Year"][::3])
ax.set_xlabel("Año de matriculación")
ax.set_ylabel("Precio medio (€)")
ygrid(ax)
ax.set_title("Precio medio por año")
fig.tight_layout()
plt.show()


### Dispersión


In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.6))
sns.scatterplot(
    data=muestra, x="KM", y="Price", hue="Combustible",
    palette={nombre: PAL[i % len(PAL)] for i, nombre in enumerate(combustibles)},
    s=28, edgecolor=SURFACE, linewidth=0.8, ax=ax,
)
ax.set_xlabel("Kilómetros")
ax.set_ylabel("Precio (€)")
ax.set_title("Kilómetros frente a precio")
ax.legend(title="Combustible", loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()


### Histograma


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.8))
sns.histplot(data=df, x="Price", bins=30, color=PAL[0], edgecolor=SURFACE, linewidth=1.2, ax=ax)
ax.set_xlabel("Precio (€)")
ax.set_ylabel("Coches")
ygrid(ax)
ax.set_title("Distribución de precios")
fig.tight_layout()
plt.show()


### Tarta


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 5.4))
ax.pie(
    por_combustible.values,
    labels=[f"{nombre}\n{valor}" for nombre, valor in por_combustible.items()],
    colors=sns.color_palette(PAL[: len(por_combustible)]),
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": SURFACE, "linewidth": 2},
    textprops={"color": TEXT, "fontsize": 8},
)
ax.axis("equal")
ax.grid(False)
ax.set_title("Anuncios por combustible")
fig.tight_layout()
plt.show()


## 4. Creación y Despliegue de la Aplicación con Streamlit

Para interactuar con el análisis de forma dinámica, crearemos una interfaz web con Streamlit que recopila los gráficos creados anteriormente.

Ejecuta la celda de abajo para iniciar el servidor de Streamlit y generar tu enlace seguro de acceso.


In [ ]:
%%writefile app.py
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Coches usados en España", layout="centered")

SURFACE = "#fcfcfb"
TEXT = "#0b0b0b"
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
AZUL = "#2a78d6"

COMBUSTIBLE = {
    "Gasolina": "Gasolina",
    "Diésel": "Diésel",
    "Diesel": "Diésel",
    "Híbrido": "Híbrido",
    "Hibrido": "Híbrido",
    "Híbrido Enchufable": "Híbrido enchufable",
    "Eléctrico": "Eléctrico",
    "Electrico": "Eléctrico",
    "GLP": "GLP",
    "GNC": "GNC",
}

BARRAS = """
st.bar_chart(
    por_marca,
    x="Brand",
    y="Price",
    x_label="Marca",
    y_label="Precio medio (€)",
    color=AZUL,
    horizontal=True,
    sort=False,
)
"""

LINEA = """
st.line_chart(
    por_anio,
    x="Year",
    y="Price",
    x_label="Año de matriculación",
    y_label="Precio medio (€)",
    color=AZUL,
)
"""

DISPERSION = """
st.scatter_chart(
    muestra,
    x="KM",
    y="Price",
    x_label="Kilómetros",
    y_label="Precio (€)",
    color="Combustible",
)
"""

HISTOGRAMA = """
precios = (
    pd.cut(df["Price"], bins=20)
    .value_counts()
    .sort_index()
    .rename_axis("Tramo")
    .reset_index(name="Coches")
)
precios["Tramo"] = precios["Tramo"].astype(str)
st.bar_chart(
    precios,
    x="Tramo",
    y="Coches",
    x_label="Tramo de precio (€)",
    y_label="Coches",
    color=AZUL,
    sort=False,
)
"""

TARTA = """
fig, ax = plt.subplots(figsize=(5.4, 5.4))
ax.pie(
    por_combustible.values,
    labels=[f"{nombre}\\n{valor}" for nombre, valor in por_combustible.items()],
    colors=PAL[: len(por_combustible)],
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": SURFACE, "linewidth": 2},
    textprops={"color": TEXT, "fontsize": 8},
)
ax.axis("equal")
ax.set_title("Anuncios por combustible")
fig.tight_layout()
st.pyplot(fig)
"""

st.title("Coches usados en España")
st.write("Los mismos cinco gráficos, dibujados con Streamlit. Dataset: Used Cars Prices in Spain.")
st.markdown(
    "<style>[data-testid='stCode'] code { white-space: pre-wrap !important; word-break: break-word; }</style>",
    unsafe_allow_html=True,
)

ruta_csv = Path("data/Cars_Data_6k.csv")
if not ruta_csv.exists():
    candidatos = list(Path("data").glob("*.csv")) if Path("data").exists() else []
    if not candidatos:
        st.error("Falta el CSV en data/. Ejecuta antes la descarga de Kaggle en el cuaderno.")
        st.stop()
    ruta_csv = candidatos[0]

df = pd.read_csv(ruta_csv)
df.columns = [c.strip() for c in df.columns]
for col in ["Price", "KM", "CV", "Year"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["Brand", "Price", "KM", "Year", "Fuel"]).copy()
df = df[(df["Price"] > 500) & (df["Price"] < 150_000)]
df = df[(df["KM"] >= 0) & (df["KM"] < 400_000)]
df = df[(df["Year"] >= 2000) & (df["Year"] <= 2024)]
df["Combustible"] = df["Fuel"].map(COMBUSTIBLE).fillna(df["Fuel"])

top_marcas = df["Brand"].value_counts().head(10).index
por_marca = (
    df[df["Brand"].isin(top_marcas)]
    .groupby("Brand", as_index=False)["Price"]
    .mean()
    .sort_values("Price", ascending=False)
)
por_anio = df.groupby("Year", as_index=False)["Price"].mean().sort_values("Year")
muestra = df.sample(n=min(800, len(df)), random_state=42)
por_combustible = df["Combustible"].value_counts()

def seccion(nombre, que_es, codigo):
    st.header(nombre)
    st.write(que_es)
    exec(codigo, {
        "st": st, "plt": plt, "pd": pd, "df": df, "AZUL": AZUL, "PAL": PAL,
        "SURFACE": SURFACE, "TEXT": TEXT,
        "por_marca": por_marca, "por_anio": por_anio, "muestra": muestra,
        "por_combustible": por_combustible,
    })
    st.code(codigo.strip(), language="python")

seccion("Barras", "Precio medio de las 10 marcas con más anuncios, con st.bar_chart.", BARRAS)
seccion("Línea", "Precio medio según el año de matriculación, con st.line_chart.", LINEA)
seccion("Dispersión", "Kilómetros frente a precio, coloreado por combustible, con st.scatter_chart.", DISPERSION)
seccion("Histograma", "Distribución de precios. Streamlit no tiene histograma: se usa st.bar_chart por tramos.", HISTOGRAMA)
seccion("Tarta", "Reparto por combustible. Streamlit no tiene tarta: se muestra con st.pyplot.", TARTA)


In [ ]:
import subprocess
import time
from google.colab import output

# 1. Asegurar que no hay procesos de Streamlit previos activos
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
time.sleep(2)

# 2. Iniciar Streamlit en segundo plano apuntando al puerto local 8501
print("Iniciando el servidor de Streamlit...")
log_st = open("streamlit.log", "w")
subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false"
    ],
    stdout=log_st,
    stderr=subprocess.STDOUT,
)
time.sleep(4)

# 3. Generar el enlace seguro oficial de Google Colab de forma nativa
enlace_seguro = output.eval_js("google.colab.kernel.proxyPort(8501)")
print(f"\n👉 La aplicación está lista. Accede haciendo clic aquí:\n{enlace_seguro}")
